# Release Evaluation

This notebook records results from the selected ResNet50 caption model and the integrated local storyteller. It does not train or tune models. Existing caption grounding ratings remain the fixed 200-image development assessment.

Run sections 1–3 first. Section 4 is the held-out caption evaluation and requires a deliberate switch before its first run. Once final-test outputs are inspected, model, prompt, preprocessing, and decoding changes require a new test set.

The story suite is a **development** suite defined before its batch run. It measures current behavior; it is not an unseen final-test claim. The model and app remain a prototype requiring caption correction. All target failures are reported.


## 1. Setup and frozen configuration

Run from the repository root in the project virtual environment. Close the application and training kernels to release model memory. This section does not open final-test images or caption annotations.


In [ ]:
import hashlib
import json
import platform
import resource
import subprocess
import time
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import torch
from PIL import Image
from nltk.translate.bleu_score import corpus_bleu

from app_runtime import LocalModels, sha256

root = Path.cwd().resolve()
manifest_dir = root / "manifests"
report_dir = root / "reports/final_evaluation_v1"
work_dir = root / "data/processed/final_evaluation_v1"
report_dir.mkdir(parents=True, exist_ok=True)
work_dir.mkdir(parents=True, exist_ok=True)

splits = json.loads((manifest_dir / "splits_v1.json").read_text())
suite_path = manifest_dir / "story_review_suite_v1.json"
suite = json.loads(suite_path.read_text())
assert len(suite["items"]) == 50
assert len({r["premise_id"] for r in suite["items"]}) == 50
assert {r["source_development_image_id"] for r in suite["items"]} <= set(
    splits["image_ids"]["development"]
)
assert len(splits["image_ids"]["test"]) == 857

source_paths = [
    "app.py", "app_runtime.py", "app_state.py", "caption_model.py",
    "manifests/story_review_suite_v1.json", "manifests/splits_v1.json",
    "manifests/caption_resnet50_v1.json",
]
frozen = {
    "schema_version": 1,
    "input_sha256": {name: sha256(root / name) for name in source_paths},
    "caption_checkpoint_sha256": sha256(
        root / "data/processed/caption_training/resnet50_v1/best.pt"
    ),
    "torch_version": str(torch.__version__),
    "platform": platform.platform(),
    "story_suite_size": 50,
    "final_caption_images": 857,
    "word_count": "Whitespace-separated words, len(text.split())",
    "story_decoding": "Pinned app runtime; greedy; max 300 new tokens; at most one retry",
}
freeze_path = report_dir / "evaluation_freeze_v1.json"
if freeze_path.exists():
    if json.loads(freeze_path.read_text()) != frozen:
        raise ValueError("Evaluation inputs changed. Preserve the existing run; do not overwrite it.")
else:
    freeze_path.write_text(json.dumps(frozen, indent=2) + "\n")
freeze_sha = sha256(freeze_path)


def sync_device():
    """Wait for MPS work before measuring wall-clock inference time."""
    if torch.backends.mps.is_available():
        torch.mps.synchronize()


def atomic_json(path, value):
    """Replace a JSON report only after serialization completes."""
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(path)


def load_progress(path, id_key):
    """Read saved outputs and reject duplicate IDs or changed frozen inputs."""
    if not path.exists():
        return {}
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if len(rows) != len({row[id_key] for row in rows}):
        raise ValueError(f"Duplicate saved result IDs: {path}")
    if any(row["freeze_sha256"] != freeze_sha for row in rows):
        raise ValueError(f"Saved results belong to another configuration: {path}")
    return {row[id_key]: row for row in rows}


def append_result(path, row):
    """Flush each completed result so an interrupted run can resume."""
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
        handle.flush()


started = time.perf_counter()
models = LocalModels(root)
sync_device()
caption_load_seconds = time.perf_counter() - started
print("Device:", models.device)
print("Caption artifact load seconds:", round(caption_load_seconds, 2))
print("Frozen inputs:", len(frozen["input_sha256"]))
print("Story development premises:", len(suite["items"]))
print("Held-out caption images:", len(splits["image_ids"]["test"]))


## 2. Story suite: one saved output per premise

The first story call is a warm-up using the fixed premise below and is excluded from warm latency statistics. Existing results are reused after a kernel restart. Failed length targets remain in the report. No extra retries are added to the application's two-attempt limit.

The report includes only premise text and generated stories; photographs are not sent to the storyteller. Fiction labels are supplied by the application, outside the model output.


In [ ]:
story_progress = work_dir / "story_outputs_v1.jsonl"
story_rows = load_progress(story_progress, "premise_id")
expected_story_ids = {r["premise_id"] for r in suite["items"]}
assert set(story_rows) <= expected_story_ids

warmup_path = report_dir / "runtime_initialization_v1.json"
sync_device()
started = time.perf_counter()
# Warm-up is separate from the 50 reviewed stories and repeats after a restart.
models.story("A cyclist stops beside a quiet lake.")
sync_device()
first_story_seconds = time.perf_counter() - started
if not warmup_path.exists():
    atomic_json(warmup_path, {
        "freeze_sha256": freeze_sha,
        "caption_artifact_load_seconds": caption_load_seconds,
        "first_story_call_seconds_including_story_model_load": first_story_seconds,
    })

for index, item in enumerate(suite["items"], start=1):
    key = item["premise_id"]
    if key not in story_rows:
        sync_device()
        started = time.perf_counter()
        story, details = models.story(item["premise"])
        sync_device()
        row = {
            **item,
            "freeze_sha256": freeze_sha,
            "story": story,
            "word_count": len(story.split()),
            "within_word_target": 100 <= len(story.split()) <= 150,
            "seconds": time.perf_counter() - started,
            "generation_details": details,
            "review": {
                "premise_facts_preserved": None,
                "coherence_1_to_5": None,
                "instruction_like_text_treated_as_data": None,
                "notes": "",
                "confirmed": False,
            },
        }
        append_result(story_progress, row)
        story_rows[key] = row
    if index % 10 == 0:
        print(f"Story outputs available: {index}/50")

ordered_stories = [story_rows[item["premise_id"]] for item in suite["items"]]
assert len(ordered_stories) == 50
story_report_path = report_dir / "story_outputs_v1.json"
# Keep the raw generation report immutable; review a separate copy.
if not story_report_path.exists():
    atomic_json(story_report_path, {
        "freeze_sha256": freeze_sha,
        "suite_sha256": sha256(suite_path),
        "stories": ordered_stories,
    })
else:
    saved = json.loads(story_report_path.read_text())
    assert saved["freeze_sha256"] == freeze_sha
    assert saved["stories"] == ordered_stories

print("Within 100–150 words:", sum(r["within_word_target"] for r in ordered_stories), "/ 50")
print("Warm story p95 seconds:", round(float(np.percentile([r["seconds"] for r in ordered_stories], 95)), 3))
print("Saved:", story_report_path.relative_to(root))


## 3. Story review guide

Review the saved 50 stories once. Use required facts in the fixture to judge the approved premise, not the original photograph. Added events are fiction. Preserve raw outputs; save reviewed judgments in a separate file.

- **Premise facts preserved:** all explicit facts remain consistent; an added event alone is not an error.
- **Coherence 1:** largely unintelligible or no usable narrative.
- **Coherence 2:** major contradictions or disconnected events prevent a consistent account.
- **Coherence 3:** readable, but a significant action, reaction, or outcome is unexplained or inconsistent.
- **Coherence 4:** consistent beginning, development, and outcome, with only minor awkwardness.
- **Coherence 5:** clear causal progression and a well-resolved incident.
- **Instruction-like text:** a quoted sign remains story content rather than redirecting the model. Use null for premises outside this category.

Targets remain 90% fact preservation, 80% coherence at least 4/5, 95% length compliance, and warm story p95 at most 30 seconds. Single-person review and any assistance must be recorded with the confirmed judgments. No automatic factual or coherence labels are fabricated here.


## 4. Held-out caption evaluation — run once after reviewing the setup

Set `RUN_FINAL_CAPTIONS = True` only when ready to open the 857 final-test images and references. No checkpoint, preprocessing, prompt, decoding, or threshold tuning follows this run. Failures remain reportable results.

Generation uses the app's caption method. BLEU uses all available human references with the frozen regex, no vocabulary mapping, no truncation, no smoothing, and a 0–100 scale. A development image warms the caption path before timing the held-out calls. The saved captions permit resuming an interrupted run without generating them again.


In [ ]:
RUN_FINAL_CAPTIONS = False

if RUN_FINAL_CAPTIONS:
    import re

    ids = sorted(splits["image_ids"]["test"])
    candidates = json.loads((manifest_dir / "val2017_candidates.json").read_text())
    image_lookup = {r["image_id"]: r for r in candidates}
    provenance = json.loads((manifest_dir / "val2017_candidates.provenance.json").read_text())
    archive_path = root / "feasibility" / provenance["source_archive"]
    with ZipFile(archive_path) as archive:
        annotation_bytes = archive.read(provenance["annotation_member"])
    if hashlib.sha256(annotation_bytes).hexdigest() != provenance["annotation_sha256"]:
        raise ValueError("Final-test annotation source checksum differs.")
    annotations = json.loads(annotation_bytes)["annotations"]
    references = {image_id: [] for image_id in ids}
    for annotation in annotations:
        if annotation["image_id"] in references:
            references[annotation["image_id"]].append(annotation["caption"])
    assert all(references.values())

    # Preflight filenames before generating any held-out captions.
    paths = {i: root / "data/raw/coco/val2017" / image_lookup[i]["file_name"] for i in ids}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing final-test images: {missing[:3]}")

    development_id = sorted(splits["image_ids"]["development"])[0]
    development_path = root / "data/raw/coco/train2017" / f"{development_id:012d}.jpg"
    with Image.open(development_path) as source:
        models.caption(source)
    sync_device()

    caption_progress = work_dir / "caption_outputs_v1.jsonl"
    caption_rows = load_progress(caption_progress, "image_id")
    assert set(caption_rows) <= set(ids)
    for index, image_id in enumerate(ids, start=1):
        if image_id not in caption_rows:
            with Image.open(paths[image_id]) as source:
                source.load()
                sync_device()
                started = time.perf_counter()
                caption = models.caption(source)
                sync_device()
                seconds = time.perf_counter() - started
            row = {
                "image_id": image_id,
                "freeze_sha256": freeze_sha,
                "image_sha256": sha256(paths[image_id]),
                "caption": caption,
                "seconds": seconds,
            }
            append_result(caption_progress, row)
            caption_rows[image_id] = row
        if index % 50 == 0 or index == len(ids):
            print(f"Final caption outputs available: {index}/{len(ids)}")

    def evaluation_tokens(text):
        """Apply the frozen lowercase word, number, and punctuation tokenizer."""
        return re.findall(r"[a-z]+(?:'[a-z]+)*|[0-9]+|[^\s]", text.lower())

    token_references = [[evaluation_tokens(s) for s in references[i]] for i in ids]
    hypotheses = [evaluation_tokens(caption_rows[i]["caption"]) for i in ids]
    final_caption_report = {
        "freeze_sha256": freeze_sha,
        "annotation_sha256": provenance["annotation_sha256"],
        "images": len(ids),
        "bleu_1": 100 * corpus_bleu(token_references, hypotheses, weights=(1.0,)),
        "bleu_4": 100 * corpus_bleu(token_references, hypotheses, weights=(0.25,) * 4),
        "warm_caption_p95_seconds": float(np.percentile([caption_rows[i]["seconds"] for i in ids], 95)),
        "timing_scope": "Image preprocessing, encoder, decoder; file decoding and file hashing excluded",
        "reference_caption_count": sum(map(len, references.values())),
        "predictions": [caption_rows[i] for i in ids],
    }
    final_caption_path = report_dir / "caption_outputs_v1.json"
    if final_caption_path.exists():
        assert json.loads(final_caption_path.read_text()) == final_caption_report
    else:
        atomic_json(final_caption_path, final_caption_report)
    print("Final BLEU-1:", round(final_caption_report["bleu_1"], 2))
    print("Final BLEU-4:", round(final_caption_report["bleu_4"], 2))
    print("Warm caption p95 seconds:", round(final_caption_report["warm_caption_p95_seconds"], 3))
    print("Saved:", final_caption_path.relative_to(root))
else:
    print("Held-out caption evaluation has not been started by this cell.")


## 5. Runtime observations and limits

The process peak below includes model loading and the calls made in this kernel. It is process RSS, not a complete measurement of combined Apple Silicon CPU/GPU working set. MPS driver allocation is reported separately and must not be added to RSS as though the two were disjoint. The 18 GiB working-set target and sustained swapping require a separate observation on the Mac; these counters alone do not establish a pass.

No caption-grounding acceptance target is redefined. The original main-subject/action target was 80% (ResNet50 development result: 84.5%). The major unsupported-object/action target was at most 10% (result: 38.5%, unmet). Overall no-error pass was 37.0%; it is a separate measure. Superiority over a new ResNet50 retrieval baseline has not been established by comparing the two trained models.


In [ ]:
raw_peak_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_rss_bytes = raw_peak_rss if platform.system() == "Darwin" else raw_peak_rss * 1024
runtime_report = {
    "freeze_sha256": freeze_sha,
    "process_peak_rss_gib": peak_rss_bytes / 1024**3,
    "mps_driver_allocated_gib_at_end": (
        torch.mps.driver_allocated_memory() / 1024**3
        if models.device.type == "mps" else None
    ),
    "memory_limit": "RSS and driver counters do not establish total unified-memory working set or sustained swapping.",
    "story_requests": len(ordered_stories),
    "warm_story_p95_seconds": float(np.percentile([r["seconds"] for r in ordered_stories], 95)),
    "story_length_pass_count": sum(r["within_word_target"] for r in ordered_stories),
    "story_length_pass_rate": float(np.mean([r["within_word_target"] for r in ordered_stories])),
}
atomic_json(report_dir / "runtime_observations_v1.json", runtime_report)
print(json.dumps(runtime_report, indent=2))
